# Hangman — AI Edition

เกม Hangman 3 โหมด:
- **Classic** — ผู้เล่นเดาเอง ไม่มี AI
- **AI Assist** — ผู้เล่นเดาเอง แต่ AI แนะนำ top-5 ตัวอักษรก่อนทุก turn
- **VS AI** — ระบบสุ่มคำเดียวกันให้ทั้งคู่ ผู้เล่นและ AI สลับ turn เดา ใครผิดครบ 6 ก่อน = แพ้

> ต้องรัน cell แรกก่อนเสมอ เพื่อโหลด imports, helpers และ word list

In [ ]:
import random
import sys
import os

sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
from predict import predict

# ============================================================
# ASCII Art
# ============================================================

HANGMAN_STAGES = [
    """
  +---+
  |   |
      |
      |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
      |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
  |   |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|   |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
 /    |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
 / \\  |
      |
========="""
]

MAX_WRONG = len(HANGMAN_STAGES) - 1
W = 40

# ============================================================
# Word List
# ============================================================

WORD_CATEGORIES = {
    "animals":   ["elephant", "giraffe", "penguin", "dolphin", "cheetah",
                  "kangaroo", "crocodile", "butterfly", "octopus", "flamingo"],
    "fruits":    ["strawberry", "pineapple", "watermelon", "blueberry", "mango",
                  "avocado", "raspberry", "pomegranate", "apricot", "coconut"],
    "countries": ["thailand", "australia", "brazil", "canada", "germany",
                  "japan", "mexico", "norway", "portugal", "sweden"],
}

def choose_word():
    category = random.choice(list(WORD_CATEGORIES.keys()))
    word = random.choice(WORD_CATEGORIES[category])
    return word, category

# ============================================================
# Pretty Print Helpers
# ============================================================

def header(title):
    print("=" * W)
    print(title.center(W))
    print("=" * W)

def divider():
    print("-" * W)

def display_word(word, guessed):
    return "  ".join(c.upper() if c in guessed else "_" for c in word)

def print_state(label, word, guessed, wrong_count):
    divider()
    print(f"  {label}")
    divider()
    print(HANGMAN_STAGES[wrong_count])
    print()
    print(f"  Word    :  {display_word(word, guessed)}")
    wrong_letters = sorted(guessed - set(word))
    wrong_str = "  ".join(wrong_letters).upper() if wrong_letters else "-"
    lives = MAX_WRONG - wrong_count
    print(f"  Wrong   :  {wrong_str}")
    print(f"  Lives   :  {'[ ]' * lives}{'[X]' * wrong_count}  ({wrong_count}/{MAX_WRONG})")
    divider()

## Mode 1 — Classic Hangman
ผู้เล่นเดาเอง ไม่มี AI

In [ ]:
def get_guess(guessed_letters):
    """รับ input ตัวอักษรจากผู้เล่น พร้อม validation"""
    while True:
        guess = input("Guess a letter: ").strip().lower()
        if len(guess) != 1:
            print("Please enter exactly one letter.")
        elif not guess.isalpha():
            print("Please enter a letter (a-z).")
        elif guess in guessed_letters:
            print(f"You already guessed '{guess}'. Try a different letter.")
        else:
            return guess

def play_hangman():
    """ฟังก์ชันหลักสำหรับเล่นเกม Hangman"""
    print("=" * 40)
    print("       Welcome to HANGMAN!")
    print("=" * 40)

    word, category = choose_word()
    guessed_letters = set()
    wrong_guesses = 0

    print(f"Category: {category.upper()}")
    print(f"The word has {len(word)} letters.\n")

    while wrong_guesses < MAX_WRONG:
        print(HANGMAN_STAGES[wrong_guesses])
        print(f"\nWord: {display_word(word, guessed_letters)}")
        print(f"Wrong guesses ({wrong_guesses}/{MAX_WRONG}): "
              f"{', '.join(sorted(guessed_letters - set(word))) or '-'}")

        if all(letter in guessed_letters for letter in word):
            print(f"\nYou WIN! The word was '{word}'.")
            return

        guess = get_guess(guessed_letters)
        guessed_letters.add(guess)

        if guess in word:
            print(f"'{guess}' is in the word!")
        else:
            wrong_guesses += 1
            print(f"'{guess}' is NOT in the word.")

    print(HANGMAN_STAGES[MAX_WRONG])
    print(f"\nGame Over! The word was '{word}'.")


play_hangman()

## Mode 2 — AI Assist
ผู้เล่นเดาเอง แต่ AI แนะนำ top-5 ตัวอักษรก่อนทุก turn

In [ ]:
def play_ai_assist():
    # แสดงหัวข้อเกม
    header("HANGMAN  —  AI ASSIST MODE")

    # สุ่มคำศัพท์และหมวดหมู่
    word, category = choose_word()
    guessed = set()
    wrong_count = 0

    print(f"  Category : {category.upper()}")
    print(f"  Letters  : {len(word)}")
    divider()

    # วนลูปจนกว่าจะชนะหรือเดาผิดครบ
    while wrong_count < MAX_WRONG:

        # สร้างรูปแบบคำที่เดาได้
        pattern = "".join(c if c in guessed else "_" for c in word)

        # ชนะเมื่อไม่มี "_" เหลือ
        if "_" not in pattern:
            print_state("YOUR BOARD", word, guessed, wrong_count)
            print(f"  RESULT  :  YOU WIN!")
            print(f"  Word was:  {word.upper()}")
            divider()
            return

        # แสดงสถานะเกม
        print_state("YOUR BOARD", word, guessed, wrong_count)

        # เตรียมข้อมูลให้ AI แนะนำ
        wrong_set = guessed - set(word)
        suggestions = predict(pattern, guessed & set(word), wrong_set)

        print(f"  AI Hint  :  {' | '.join(s.upper() for s in suggestions)}")
        divider()

        # รับตัวอักษรจากผู้เล่น
        while True:
            guess = input("  Your guess > ").strip().lower()

            if len(guess) != 1 or not guess.isalpha():
                print("  Enter a single letter (a-z).")
            elif guess in guessed:
                print(f"  '{guess.upper()}' already guessed. Try again.")
            else:
                break

        guessed.add(guess)

        # ตรวจสอบว่าทายถูกหรือไม่
        if guess in word:
            print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
        else:
            wrong_count += 1
            print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    # แสดงผลเมื่อแพ้
    print_state("YOUR BOARD", word, guessed, wrong_count)
    print(f"  RESULT  :  GAME OVER")
    print(f"  Word was:  {word.upper()}")
    divider()


# เล่นเกมซ้ำจนกว่าผู้เล่นจะออก
while True:

    play_ai_assist()

    if input("\n  Play again? (yes/no) > ").strip().lower() != "yes":
        print("  Thanks for playing! Goodbye!")
        divider()
        break

## Mode 3 — VS AI
ระบบสุ่มคำเดียวกันให้ทั้งคู่ ผู้เล่นและ AI สลับ turn เดา ใครผิดครบ 6 ก่อน = แพ้

**กติกา:**
- ระบบสุ่มคำมาให้ทั้งคู่ (คำเดียวกัน)
- ผู้เล่นและ AI เดาแยกกันสนิท ไม่เห็น guessed ของอีกฝ่าย
- สลับ turn ไปเรื่อยๆ จนฝ่ายใดฝ่ายหนึ่งผิดครบ 6 ครั้ง

In [ ]:
def vs_ai_turn_player(word, guessed, wrong_count):
    # สร้างรูปแบบคำปัจจุบัน
    pattern = "".join(c if c in guessed else "_" for c in word)

    # จบตาถ้าชนะหรือแพ้แล้ว
    if "_" not in pattern or wrong_count >= MAX_WRONG:
        return guessed, wrong_count, True

    # แสดงกระดานของผู้เล่น
    print_state("YOUR TURN  —  Guess the word", word, guessed, wrong_count)

    # รับตัวอักษรจากผู้เล่น
    while True:
        guess = input("  Your guess > ").strip().lower()

        if len(guess) != 1 or not guess.isalpha():
            print("  Enter a single letter (a-z).")
        elif guess in guessed:
            print(f"  '{guess.upper()}' already guessed. Try again.")
        else:
            break

    guessed.add(guess)

    # ตรวจสอบว่าทายถูกหรือไม่
    if guess in word:
        print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
    else:
        wrong_count += 1
        print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    done = "_" not in "".join(c if c in guessed else "_" for c in word) or wrong_count >= MAX_WRONG

    return guessed, wrong_count, done


def vs_ai_turn_ai(word, guessed, wrong_count):
    # สร้างรูปแบบคำปัจจุบัน
    pattern = "".join(c if c in guessed else "_" for c in word)

    # จบตาถ้าชนะหรือแพ้แล้ว
    if "_" not in pattern or wrong_count >= MAX_WRONG:
        return guessed, wrong_count, True

    # แสดงกระดานของ AI
    print_state("AI TURN  —  AI guesses the word", word, guessed, wrong_count)

    # ให้ AI ทำนายตัวอักษร
    wrong_set = guessed - set(word)
    suggestions = predict(pattern, guessed & set(word), wrong_set)
    guess = suggestions[0]

    print(f"  AI Rank  :  {' | '.join(s.upper() for s in suggestions)}")
    print(f"  AI Guess :  {guess.upper()}")
    divider()
    input("  Press Enter to continue...")

    guessed.add(guess)

    # ตรวจสอบผลการทายของ AI
    if guess in word:
        print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
    else:
        wrong_count += 1
        print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    done = "_" not in "".join(c if c in guessed else "_" for c in word) or wrong_count >= MAX_WRONG

    return guessed, wrong_count, done


def print_result(winner, word, p_wrong, a_wrong):
    # แสดงผลการแข่งขัน
    divider()
    print("  GAME OVER".center(W))
    divider()

    print(f"  The word   :  {word.upper()}")
    print(f"  Your wrong :  {p_wrong}/{MAX_WRONG}")
    print(f"  AI wrong   :  {a_wrong}/{MAX_WRONG}")

    divider()
    print(f"  WINNER  :  {winner}".center(W))
    divider()


def play_vs_ai():
    # เริ่มโหมดแข่งขันกับ AI
    header("HANGMAN  —  VS AI MODE")

    word, category = choose_word()

    print(f"  Category : {category.upper()} ({len(word)} letters)")
    print(f"  Same word for both — may the best guesser win!")
    divider()

    # สถานะของผู้เล่นและ AI
    p_guessed, p_wrong = set(), 0
    a_guessed, a_wrong = set(), 0

    # เล่นสลับตาระหว่างผู้เล่นและ AI
    while True:

        # ตาผู้เล่น
        p_guessed, p_wrong, p_done = vs_ai_turn_player(word, p_guessed, p_wrong)

        if p_wrong >= MAX_WRONG:
            print_result("AI", word, p_wrong, a_wrong)
            return

        if p_done:
            print_result("YOU", word, p_wrong, a_wrong)
            return

        # ตา AI
        a_guessed, a_wrong, a_done = vs_ai_turn_ai(word, a_guessed, a_wrong)

        if a_wrong >= MAX_WRONG:
            print_result("YOU", word, p_wrong, a_wrong)
            return

        if a_done:
            print_result("AI", word, p_wrong, a_wrong)
            return


# เล่นเกมซ้ำจนกว่าผู้เล่นจะออก
while True:

    play_vs_ai()

    if input("\n  Play again? (yes/no) > ").strip().lower() != "yes":
        print("  Thanks for playing! Goodbye!")
        divider()
        break